# Nocarz — Ewaluacja eksperymentu A/B

Analiza loga mikroserwisu (`logs/predictions.jsonl`). Mikroserwis kieruje ruch 50/50 do modelu A i B (routing po haszu `listing_id`, deterministyczny) i loguje predykcje. Tutaj łączymy log z prawdą (`ground_truth.csv`) i oceniamy, który model jest lepszy.

> Uruchom najpierw: serwer (`scripts/run_server.ps1`) → `scripts/simulate_clients.py --n 1000 --paired`.

In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
ROOT = next((q for q in [_p, *_p.parents] if (q / 'src' / 'nocarz').exists()), _p)
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT / 'scripts'))
import pandas as pd, matplotlib.pyplot as plt
import evaluate_ab as ab
log = ab.load_log(); df_all = ab.join_truth(log); nat = ab.natural_traffic(df_all)
print('Rekordow w logu:', len(log), '| z prawda:', len(df_all), '| ruch naturalny A/B:', len(nat))
log[['timestamp','assigned_model','assignment_reason','model_version','listing_id','predicted_annual_revenue']].head()

Rekordow w logu: 3000 | z prawda: 3000 | ruch naturalny A/B: 1000


,timestamp,assigned_model,assignment_reason,model_version,listing_id,predicted_annual_revenue
0,2026-06-03 12:53:46.457025+00:00,b,hash,HistGradientBoostingRegressor-revenue-2026.06.03,945600815997104634,42298.629132
1,2026-06-03 12:53:46.459535+00:00,a,forced,DistrictMeanRegressor-revenue-2026.06.03,945600815997104634,56251.399209
2,2026-06-03 12:53:46.464642+00:00,b,forced,HistGradientBoostingRegressor-revenue-2026.06.03,945600815997104634,42298.629132
3,2026-06-03 12:53:46.466310+00:00,a,hash,DistrictMeanRegressor-revenue-2026.06.03,6641918,48976.273275
4,2026-06-03 12:53:46.467723+00:00,a,forced,DistrictMeanRegressor-revenue-2026.06.03,6641918,48976.273275


## 1. Podział ruchu A/B (kontrola routingu)

In [2]:
print(nat['assigned_model'].value_counts())
print('\nWersje modeli w logu:'); print(nat.groupby('assigned_model')['model_version'].unique())

assigned_model
b    536
a    464
Name: count, dtype: int64

Wersje modeli w logu:
assigned_model
a           [DistrictMeanRegressor-revenue-2026.06.03]
b    [HistGradientBoostingRegressor-revenue-2026.06...
Name: model_version, dtype: object


## 2. Metryki per model (ruch naturalny)

In [3]:
metrics = ab.per_model_metrics(nat, ab.REVENUE)
occ_metrics = ab.per_model_metrics(nat, ab.OCCUPANCY)
print('Oblozenie (drugi wynik Canvas):'); print(occ_metrics.to_string(index=False))
print('\nPrzychod (glowne KPI):')
metrics

Oblozenie (drugi wynik Canvas):
model   n     rmse      mae      r2  median_ae
    a 464 0.357087 0.321268 0.03017     0.3225
    b 536 0.347257 0.303410 0.07627     0.3024

Przychod (glowne KPI):


,model,n,rmse,mae,r2,median_ae
0,a,464,46279.893257,30260.820512,0.035466,21489.936760
1,b,536,43168.190160,25229.032788,0.187210,16859.975392


## 3. Istotność statystyczna
Testy niezależne na błędzie bezwzględnym (różne oferty w grupach A i B) + bootstrap CI dla luki RMSE. Dodatkowo test **parowany** (te same oferty przez wymuszone `/a` i `/b`) — najsilniejszy, bo eliminuje wpływ doboru ofert.

In [4]:
sig = ab.significance(nat); ci = ab.bootstrap_rmse_gap(nat); paired = ab.paired_test(df_all)
print(f"Mann-Whitney U p = {sig.get('mwu_p'):.4g}")
print(f"Welch t-test  p = {sig.get('welch_p'):.4g}")
print(f"Bootstrap 95% CI RMSE(A)-RMSE(B): [{ci[0]:,.0f}, {ci[1]:,.0f}] EUR")
if paired:
    print(f"\nTest parowany (n={paired['n_pairs']}): |blad| A={paired['mean_ae_a']:,.0f}, B={paired['mean_ae_b']:,.0f}")
    print(f"Wilcoxon p = {paired['wilcoxon_p']:.4g}; t-parowany p = {paired['paired_t_p']:.4g}")

Mann-Whitney U p = 4.288e-05
Welch t-test  p = 0.02382
Bootstrap 95% CI RMSE(A)-RMSE(B): [-8,337, 14,454] EUR

Test parowany (n=1000): |blad| A=29,797, B=25,313
Wilcoxon p = 2.944e-22; t-parowany p = 1.265e-16


## 4. Wykresy

In [5]:
paths = ab.make_plots(nat)
from matplotlib import image as mpimg
for p in paths:
    fig, axx = plt.subplots(figsize=(9, 5)); axx.imshow(mpimg.imread(p)); axx.axis('off'); plt.show()

/tmp/claude-1000/ipykernel_12005/309932011.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig, axx = plt.subplots(figsize=(9, 5)); axx.imshow(mpimg.imread(p)); axx.axis('off'); plt.show()


## 5. Werdykt

In [6]:
occ_sig = ab.significance(nat, ab.OCCUPANCY); occ_paired = ab.paired_test(df_all, ab.OCCUPANCY)
verdict = ab.decide(metrics, ci, sig, paired)
print(verdict)
ab.write_report(metrics, sig, ci, paired, verdict, occ_metrics, occ_sig, occ_paired)
print('\nZapisano raport -> reports/ab_report.md')

WYGRYWA model B (HGB): MAE 25,229 < 30,261 EUR, RMSE 43,168 < 46,280; różnica istotna (parowany Wilcoxon p=2.9e-22, Mann-Whitney p=4.3e-05). Rekomendacja: wdrożyć B (uwaga: luka RMSE nieistotna, 95% CI = [-8,337, 14,454] — RMSE zdominowane przez ciężki ogon błędów; przewaga B dotyczy ofert typowych).

Zapisano raport -> reports/ab_report.md
